In [ ]:
# Importing libaries
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shap
import tensorflow as tf
import joblib
from tensorflow.keras.preprocessing.image import ImageDataGenerator

sys.path.insert(0, os.path.abspath(".."))

from src.SHAP_analysis import 
(
    explain_cnn, 
    explain_random_forest,
    explain_rf_beeswarm
)

np.random.seed(42)
tf.random.set_seed(42)

from PIL import Image as PILImage

In [ ]:
#Rebuilding the test generator
IMG_SIZE = (224, 224)
BATCH = 32
DATA_DIR = "data/images"
TEST_DIR = "data/test_images"

# ── Training generator (used to build SHAP background for CNN) ────────────
train_datagen = ImageDataGenerator
(
    rescale = 1.0 / 225, 
    validation_split = 0.15
)

train_gen = train_datagen.flow_from_directory 
(
    DATA_DIR,
    target_size = IMG_SIZE,
    batch_size = BATCH,
    class_mode = "binary",
    subset = "training",
    shuffle = True,
    seed = 42
)

# ── Test generator ────────────────────────────────────────────────────────
test_datagen = ImageDataGenerator(rescale = 1.0 / 225)

test_gen = test_datagen.flow_from_directory
(
   TEST_DIR,
    target_size = IMG_SIZE,
    batch_size = BATCH,
    class_mode = "binary",
    shuffle = False,  
)
print("Class indices:", test_gen.class_indices)
print(f"Training batches: {len(train_gen)} ({train_gen.samples} samples)")
print(f"Test batches: {len(test_gen)} ({test_gen.samples} samples)")

In [ ]:
# Displaying the saved CNN SHAP heatmap
cnn_shap_path = PILImage.open(CNN_SHAP_OUTPUT)

fig, ax = plt.subplots(figsize = (14,5))
ax.imshow(cnn_shap_img)
ax.axis("off")
ax.set_title
(
    "CNN SHAP Heatmaps - MobileNetV2\n "
    "Red = drives 'Sold Animal' | Blue = drives 'Not Listed'"
    fontsize = 13
)
plt.tight_layout()
plt.show()

print("""
Interpretation guide:
  • If the model is working correctly, RED regions should cover the animal's
    body (fur texture, body shape, distinctive markings).
  • Blue regions covering the animal or red regions in the background suggest
    the model may be using spurious features — a sign of overfitting or
    dataset bias (e.g. all lion photos have a similar background).
""")

In [ ]:
# running RF SHAP Bar chart
RF_MODEL_PATH = "artifacts/RandomForest_HOG.pkl"
X_TEST_PATH = "artifacts/X_test_pca.npy"
RF_SHAP_OUTPUT = "artifacts/shap_rf.png"
N_RF_SAMPLES = 50
N_TOP_FEATURES = 20

explain_random_forest
(
    model_path     = RF_MODEL_PATH,
    X_test_path    = X_TEST_PATH,
    n_samples      = N_RF_SAMPLES,
    n_top_features = N_TOP_FEATURES,
    output_path    = RF_SHAP_OUTPUT,
    show           = True,
)

In [ ]:
# Displing saved RF SHAP Bar Chart
rf_shap_img = PILImage.open(RF_SHAP_OUTPUT)

fig, ax = plt.subplots(figsize = (10, 7))
ax.imshow(rf_shap_img)
ax.axis("off")
ax.set_title
(
    "Random Forest SHAP — Top HOG/PCA Feature Importances\n"
    "Mean |SHAP value| for 'Sold Animal' class",
    fontsize=13
)
plt.tight_layout()
plt.show()

print("""
Interpretation guide:
  • The bars represent PCA components (abstract combinations of HOG gradients).
  • A long bar for Feature N means that PCA component consistently influenced
    predictions toward or away from 'Sold Animal'.
  • This confirms the model is using real image structure (edges, textures)
    rather than random noise to make decisions.
""")

In [ ]:
# running RF SHAP beesawarm plot
RF_BEESWARM_OUTPUT = "artifacts/shap_rf_beeswarm.png"

explain_rf_beeswarm
(
    model_path     = RF_MODEL_PATH,
    X_test_path    = X_TEST_PATH,
    n_samples      = N_RF_SAMPLES,
    n_top_features = 15,             
    output_path    = RF_BEESWARM_OUTPUT,
    show           = True,
)

In [ ]:
# Bar chart vs Beeswarm 
bar_img = PILImage.open(RF_SHAP_OUTPUT)
beeswarm_img = PILImage.open(RF_BEESWARM_OUTPUT)

fig, axes = plt.subplots(1, 2, figsize = (18, 7))

axes[0].imshow(bar_img)
axes[0].axis("off")
axes[0].set_title("Bar Chart\n(Mean |SHAP|  — overall importance)", fontsize = 12)

axes[1].imshow(beeswarm_img)
axes[1].axis("off")
axes[1].set_title("Beeswarm\n(Direction + magnitude per sample)", fontsize = 12)

fig.suptitle("Random Forest SHAP — Two Views of Feature Importance", fontsize = 14, y = 1.01)
plt.tight_layout()
plt.show()

In [ ]:
# model comparison table
comparison = pd.DataFrame
({
    "Attribute":
    [
        "SHAP Method",
        "Input Features",
        "Output Type",
        "Speed",
        "Interpretability",
        "Best Use",
    ],
    "CNN (MobileNetV2)": 
    [
        "GradientExplainer",
        "Raw pixels (224×224×3)",
        "Pixel heatmap overlaid on image",
        "Slow (gradient computation per sample)",
        "Visual — non-experts can understand",
        "Proving the model looks at the animal body",
    ],
    "Random Forest + HOG":
    [
        "TreeExplainer",
        "300 PCA-reduced HOG components",
        "Feature importance bar chart / beeswarm",
        "Very fast (exact tree traversal)",
        "Technical — features are abstract PCA components",
        "Confirming model uses structural image patterns",
    ],
})

comparison = comparison.set_index("Attribute")

display(comparison.style.set_properties
(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'font-size': '12px',
}).set_table_styles
([
    {'selector': 'th', 'props': [('background-color', '#1B4332'), ('color', 'white'), ('font-size', '13px')]}
]))

In [ ]:
# checking the artifacts
expected_files = 
[
    CNN_SHAP_OUTPUT,
    RF_SHAP_OUTPUT,
    RF_BEESWARM_OUTPUT,
]

print("SHAP artifact check:")
print("-" * 45)

all_ok = True
for path in expected_files:
    exists  = os.path.isfile(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status  = f"✓  {size_kb:6.1f} KB" if exists else "✗  MISSING"
    print(f"  {status}  {path}")
    if not exists:
        all_ok = False

print("-" * 45)
print("All artifacts present" if all_ok else "Some artifacts are missing — re-run the cells above.")